# 🧪 Data Manipulation in Python — Road to Building a Model

This notebook walks you through **every data manipulation skill** you need before training a machine learning model — from raw data ingestion all the way to a trained, evaluated classifier.

### What you'll learn
| Section | Topic |
|---------|-------|
| 1 | NumPy — fast numerical arrays |
| 2 | Pandas — tabular data & DataFrames |
| 3 | Loading & inspecting real data |
| 4 | Cleaning dirty data |
| 5 | Exploratory Data Analysis (EDA) |
| 6 | Feature engineering |
| 7 | Encoding & scaling |
| 8 | Train/test split |
| 9 | Building & evaluating a model |
| 10 | End-to-end pipeline |

### Setup — run this first
```bash
pip install numpy pandas matplotlib seaborn scikit-learn
```

In [2]:
# Core imports — run this cell before anything else
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings('ignore')

# Consistent styling
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
np.random.seed(42)

print('✅ All libraries imported successfully!')

✅ All libraries imported successfully!


---
## Section 1 — NumPy: Fast Numerical Arrays

**NumPy** is the foundation of data science in Python. Almost every library (pandas, scikit-learn, TensorFlow) uses NumPy arrays under the hood.

Key idea: a `ndarray` is like a Python list but **vectorized** — operations apply to all elements at once without explicit loops.

In [3]:
# ── 1.1 Creating arrays ──────────────────────────────────────────────
a = np.array([1, 2, 3, 4, 5])          # from a list
b = np.zeros((3, 4))                    # 3x4 matrix of zeros
c = np.ones((2, 3))                     # 2x3 matrix of ones
d = np.arange(0, 10, 2)                 # [0, 2, 4, 6, 8]
e = np.linspace(0, 1, 5)               # 5 evenly spaced points
f = np.random.randn(3, 3)              # 3x3 random normal matrix

print('Array a:', a)
print('Zeros (3x4):\n', b)
print('Arange:', d)
print('Linspace:', e)
print('Random 3x3:\n', f.round(2))

Array a: [1 2 3 4 5]
Zeros (3x4):
 [[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]
Arange: [0 2 4 6 8]
Linspace: [0.   0.25 0.5  0.75 1.  ]
Random 3x3:
 [[ 0.5  -0.14  0.65]
 [ 1.52 -0.23 -0.23]
 [ 1.58  0.77 -0.47]]


In [4]:
# ── 1.2 Vectorized operations (no loops needed!) ─────────────────────
x = np.array([1, 2, 3, 4, 5], dtype=float)

print('x:       ', x)
print('x * 2:   ', x * 2)          # multiply each element
print('x ** 2:  ', x ** 2)         # square each element
print('sqrt(x): ', np.sqrt(x).round(3))
print('mean:    ', x.mean())
print('std:     ', x.std().round(3))

# Why this matters: sklearn, pandas internals all rely on this speed
big = np.random.randn(1_000_000)
%timeit big.mean()   # microseconds — try doing this with a loop!

x:        [1. 2. 3. 4. 5.]
x * 2:    [ 2.  4.  6.  8. 10.]
x ** 2:   [ 1.  4.  9. 16. 25.]
sqrt(x):  [1.    1.414 1.732 2.    2.236]
mean:     3.0
std:      1.414
896 μs ± 98 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [9]:
# ── 1.3 Indexing, slicing & boolean masks ────────────────────────────
m = np.arange(6).reshape(2, 3)   # 2D array
print('Matrix:\n', m)
print('Row 1:       ', m[1])          # second row
print('Col 2:       ', m[:, 2])       # third column
print('Submatrix:\n', m[0:2, 1:3])   # rows 0-1, cols 1-2

# Boolean mask — very common in data cleaning
data = np.array([10, -3, 7, -1, 4, -9])
print('Positive only:', data[data > 0])   # filter positives

Matrix:
 [[0 1 2]
 [3 4 5]]
Row 1:        [3 4 5]
Col 2:        [2 5]
Submatrix:
 [[1 2]
 [4 5]]
Positive only: [10  7  4]


### 🏋️ Practice 1
Create a 5×5 matrix of random integers between 1 and 100, then:
1. Print the matrix
2. Print the max value in each row
3. Normalize the matrix so all values are between 0 and 1 (hint: `(x - x.min()) / (x.max() - x.min())`)

In [87]:
# YOUR CODE HERE
r = np.random.randint(1,101, size= (5,5))

print('matrix:\n', r)
print("max\n", r.max(axis=(1)))
print("normalize\n", (r - r.min()) / (r.max() - r.min()))
print(np.ptp(r, axis=(0)))

matrix:
 [[76 69 76 15 34]
 [92 88 24  9 25]
 [77 18 53 73 40]
 [36 58 46 90 49]
 [84  8 36 15 44]]
max
 [76 92 77 90 84]
normalize
 [[0.80952381 0.72619048 0.80952381 0.08333333 0.30952381]
 [1.         0.95238095 0.19047619 0.01190476 0.20238095]
 [0.82142857 0.11904762 0.53571429 0.77380952 0.38095238]
 [0.33333333 0.5952381  0.45238095 0.97619048 0.48809524]
 [0.9047619  0.         0.33333333 0.08333333 0.42857143]]
[56 80 52 81 24]


---
## Section 2 — Pandas: Tabular Data & DataFrames

**Pandas** is how you work with structured/tabular data — think spreadsheets in Python. The two main objects are:
- `Series` — a 1D labeled array (one column)
- `DataFrame` — a 2D table (rows × columns)

In [79]:
# # ── 2.1 Creating DataFrames ──────────────────────────────────────────
# # From a dictionary
df = pd.DataFrame({
    'name':   ['Alice', 'Bob', 'Carol', 'Dave', 'Eve'],
    'age':    [25, 32, 28, 45, 22],
    'salary': [55000, 72000, 61000, 95000, 48000],
    'dept':   ['Engineering', 'Marketing', 'Engineering', 'Finance', 'Marketing']
})

print('Shape:', df.shape)   # (rows, columns)
df

Shape: (5, 4)


,name,age,salary,dept
0,Alice,25,55000,Engineering
1,Bob,32,72000,Marketing
2,Carol,28,61000,Engineering
3,Dave,45,95000,Finance
4,Eve,22,48000,Marketing


In [88]:
# ── 2.2 Selecting data ───────────────────────────────────────────────

# Single column → Series
print(df['name'])
print()

# Multiple columns → DataFrame
print(df[['name', 'salary']])
print()

# Row by label (.loc) or position (.iloc)
print(df.loc[2])           # row with index label 2
print()
print(df.iloc[0:2])        # first 2 rows by position

0    Alice
1      Bob
2    Carol
3     Dave
4      Eve
Name: name, dtype: str

    name  salary
0  Alice   55000
1    Bob   72000
2  Carol   61000
3   Dave   95000
4    Eve   48000

name            Carol
age                28
salary          61000
dept      Engineering
Name: 2, dtype: object

    name  age  salary         dept
0  Alice   25   55000  Engineering
1    Bob   32   72000    Marketing


In [89]:
# ── 2.3 Filtering rows ───────────────────────────────────────────────

# Boolean condition
print('Age > 25:')
print(df[df['age'] > 25])
print()

# Multiple conditions (use & and | not 'and'/'or')
print('Engineering dept AND salary > 55k:')
print(df[(df['dept'] == 'Engineering') & (df['salary'] > 55000)])
print()

# .query() — more readable
print(df.query('age < 30 and dept == "Marketing"'))

Age > 25:
    name  age  salary         dept
1    Bob   32   72000    Marketing
2  Carol   28   61000  Engineering
3   Dave   45   95000      Finance

Engineering dept AND salary > 55k:
    name  age  salary         dept
2  Carol   28   61000  Engineering

  name  age  salary       dept
4  Eve   22   48000  Marketing


In [90]:
# ── 2.4 Aggregation & groupby ────────────────────────────────────────

# Basic statistics
print(df[['age', 'salary']].describe())
print()

# groupby — split → apply → combine
print('Average salary by department:')
print(df.groupby('dept')['salary'].mean())
print()

# Multiple aggregations at once
print(df.groupby('dept').agg(
    headcount=('name', 'count'),
    avg_age=('age', 'mean'),
    avg_salary=('salary', 'mean')
))

             age        salary
count   5.000000      5.000000
mean   30.400000  66200.000000
std     8.961027  18349.386911
min    22.000000  48000.000000
25%    25.000000  55000.000000
50%    28.000000  61000.000000
75%    32.000000  72000.000000
max    45.000000  95000.000000

Average salary by department:
dept
Engineering    58000.0
Finance        95000.0
Marketing      60000.0
Name: salary, dtype: float64

             headcount  avg_age  avg_salary
dept                                       
Engineering          2     26.5     58000.0
Finance              1     45.0     95000.0
Marketing            2     27.0     60000.0


In [91]:
# ── 2.5 Adding & transforming columns ────────────────────────────────

df2 = df.copy()

# New column via arithmetic
df2['monthly_salary'] = df2['salary'] / 12

# New column via .apply() — for custom logic
df2['seniority'] = df2['age'].apply(lambda age: 'Senior' if age >= 30 else 'Junior')

# New column via np.where — like an IF-THEN-ELSE
df2['high_earner'] = np.where(df2['salary'] > 60000, True, False)

df2

,name,age,salary,dept,monthly_salary,seniority,high_earner
0,Alice,25,55000,Engineering,4583.333333,Junior,False
1,Bob,32,72000,Marketing,6000.000000,Senior,True
2,Carol,28,61000,Engineering,5083.333333,Junior,True
3,Dave,45,95000,Finance,7916.666667,Senior,True
4,Eve,22,48000,Marketing,4000.000000,Junior,False


### 🏋️ Practice 2
Using `df2`, answer these questions **with pandas** (no manual counting):
1. What is the highest salary in the Engineering department?
2. Add a column `'salary_rank'` that ranks employees from highest (1) to lowest salary (use `.rank(ascending=False)`)
3. How many high earners are in each department?

In [ ]:
# YOUR CODE HERE


---
## Section 3 — Loading & Inspecting Real Data

Real data comes from CSV files, databases, APIs, Excel sheets, and more. Here we'll use a built-in dataset to simulate a real-world scenario.

In [ ]:
# ── 3.1 Loading data ─────────────────────────────────────────────────
# We'll use the Titanic dataset — a classic ML classification problem
# (predict who survived based on passenger features)

url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
titanic = pd.read_csv(url)

# Common loading options:
# pd.read_csv('file.csv', sep=';', encoding='utf-8', parse_dates=['date_col'])
# pd.read_excel('file.xlsx', sheet_name='Sheet1')
# pd.read_json('file.json')

print(f'Loaded {titanic.shape[0]} rows × {titanic.shape[1]} columns')
titanic.head()

In [ ]:
# ── 3.2 First look — always do this on a new dataset ─────────────────

print('=== SHAPE ===')
print(titanic.shape)

print('\n=== COLUMN TYPES ===')
print(titanic.dtypes)

print('\n=== MISSING VALUES ===')
missing = titanic.isnull().sum()
missing_pct = (missing / len(titanic) * 100).round(1)
print(pd.concat([missing, missing_pct], axis=1, keys=['count', '%']))

print('\n=== BASIC STATS ===')
titanic.describe()

In [ ]:
# ── 3.3 Understanding categorical columns ────────────────────────────

for col in ['Survived', 'Pclass', 'Sex', 'Embarked']:
    print(f'--- {col} ---')
    print(titanic[col].value_counts())
    print()

---
## Section 4 — Cleaning Dirty Data

Real data is messy. This is usually **the most time-consuming step** in any ML project.
Common problems: missing values, wrong types, duplicates, outliers, inconsistent formatting.

In [ ]:
# ── 4.1 Handling missing values ──────────────────────────────────────
df_clean = titanic.copy()

# Strategy 1: Drop rows with missing values
# df_clean.dropna(inplace=True)   # drops ALL rows with any NaN — often too aggressive

# Strategy 2: Drop columns with too many missing values
# Cabin is 77% missing — not useful
df_clean.drop(columns=['Cabin'], inplace=True)
print('Dropped Cabin column')

# Strategy 3: Fill with median (for skewed numerical data)
age_median = df_clean['Age'].median()
df_clean['Age'].fillna(age_median, inplace=True)
print(f'Filled Age NaNs with median: {age_median}')

# Strategy 4: Fill with mode (for categorical)
embarked_mode = df_clean['Embarked'].mode()[0]
df_clean['Embarked'].fillna(embarked_mode, inplace=True)
print(f'Filled Embarked NaNs with mode: {embarked_mode}')

print(f'\nMissing values remaining: {df_clean.isnull().sum().sum()}')

In [ ]:
# ── 4.2 Fixing data types ────────────────────────────────────────────

print('Before:', df_clean.dtypes[['Survived', 'Pclass']])

# Survived and Pclass should be categories, not just integers
df_clean['Survived'] = df_clean['Survived'].astype(int)   # ensure int
df_clean['Pclass'] = df_clean['Pclass'].astype('category')

print('After: ', df_clean.dtypes[['Survived', 'Pclass']])

In [ ]:
# ── 4.3 Removing duplicates ──────────────────────────────────────────

print(f'Rows before: {len(df_clean)}')
df_clean.drop_duplicates(inplace=True)
print(f'Rows after:  {len(df_clean)}')
print(f'Duplicates removed: {titanic.shape[0] - len(df_clean)}')

In [ ]:
# ── 4.4 Detecting outliers with IQR ──────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Boxplot reveals outliers visually
df_clean[['Age', 'Fare']].boxplot(ax=axes[0])
axes[0].set_title('Boxplot — spot the outliers')

# IQR method to detect outlier rows
Q1 = df_clean['Fare'].quantile(0.25)
Q3 = df_clean['Fare'].quantile(0.75)
IQR = Q3 - Q1
outliers = df_clean[(df_clean['Fare'] < Q1 - 1.5 * IQR) | (df_clean['Fare'] > Q3 + 1.5 * IQR)]
print(f'Fare outliers detected: {len(outliers)}')
print(outliers['Fare'].describe())

# Log-transform to reduce skew (common technique)
df_clean['Fare_log'] = np.log1p(df_clean['Fare'])   # log1p = log(1 + x), safe for 0
df_clean['Fare_log'].hist(bins=30, ax=axes[1])
axes[1].set_title('Log-transformed Fare (less skewed)')

plt.tight_layout()
plt.show()

---
## Section 5 — Exploratory Data Analysis (EDA)

Before modeling, **understand your data visually**. EDA reveals patterns, relationships, and biases that drive feature engineering decisions.

In [ ]:
# ── 5.1 Target variable distribution ────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Survival rate
df_clean['Survived'].value_counts().plot(kind='bar', ax=axes[0], color=['#e74c3c', '#2ecc71'])
axes[0].set_xticklabels(['Died', 'Survived'], rotation=0)
axes[0].set_title('Survival Count')

# Age distribution
df_clean['Age'].hist(bins=30, ax=axes[1], color='steelblue', edgecolor='white')
axes[1].set_title('Age Distribution')

# Fare distribution
df_clean['Fare_log'].hist(bins=30, ax=axes[2], color='darkorange', edgecolor='white')
axes[2].set_title('Fare (log) Distribution')

plt.tight_layout()
plt.show()

In [ ]:
# ── 5.2 Survival rate by feature — spot what matters ─────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# By Sex
df_clean.groupby('Sex')['Survived'].mean().plot(kind='bar', ax=axes[0], color=['#3498db', '#e91e63'])
axes[0].set_title('Survival Rate by Sex')
axes[0].set_ylabel('Survival Rate')
axes[0].set_xticklabels(['Female', 'Male'], rotation=0)

# By Pclass
df_clean.groupby('Pclass')['Survived'].mean().plot(kind='bar', ax=axes[1], color='coral')
axes[1].set_title('Survival Rate by Class')
axes[1].set_ylabel('Survival Rate')
axes[1].set_xticklabels(['1st', '2nd', '3rd'], rotation=0)

# Age vs Survival (violin plot)
df_clean.boxplot(column='Age', by='Survived', ax=axes[2])
axes[2].set_title('Age by Survival')
axes[2].set_xlabel('Survived (0=No, 1=Yes)')

plt.tight_layout()
plt.show()
print('Key insight: Women and 1st class passengers survived at much higher rates!')

In [ ]:
# ── 5.3 Correlation heatmap ──────────────────────────────────────────
# Only numeric columns
numeric_cols = df_clean.select_dtypes(include=[np.number])

plt.figure(figsize=(10, 7))
corr = numeric_cols.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))  # hide upper triangle
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', center=0, square=True)
plt.title('Correlation Matrix')
plt.tight_layout()
plt.show()

# Survival correlations specifically
print('\nCorrelation with Survived:')
print(corr['Survived'].sort_values(ascending=False))

---
## Section 6 — Feature Engineering

Feature engineering means **creating new, informative columns from existing ones**. This is where domain knowledge pays off and often has the biggest impact on model performance.

In [ ]:
df_fe = df_clean.copy()

# ── 6.1 Extract title from Name ──────────────────────────────────────
# Name looks like: "Braund, Mr. Owen Harris" — the title (Mr, Mrs, etc.) is informative
df_fe['Title'] = df_fe['Name'].str.extract(r',\s*([^.]+)\.', expand=False).str.strip()
print('All titles found:')
print(df_fe['Title'].value_counts())

# Group rare titles
common_titles = ['Mr', 'Miss', 'Mrs', 'Master']
df_fe['Title'] = df_fe['Title'].apply(lambda t: t if t in common_titles else 'Rare')
print('\nSimplified titles:')
print(df_fe['Title'].value_counts())

In [ ]:
# ── 6.2 Family size feature ──────────────────────────────────────────
# SibSp = siblings/spouses, Parch = parents/children
df_fe['FamilySize'] = df_fe['SibSp'] + df_fe['Parch'] + 1  # +1 for themselves
df_fe['IsAlone'] = (df_fe['FamilySize'] == 1).astype(int)

print('Survival rate by family size:')
print(df_fe.groupby('FamilySize')['Survived'].mean().round(2))

In [ ]:
# ── 6.3 Age binning (discretization) ────────────────────────────────
# Instead of raw age, group into life stages
df_fe['AgeBand'] = pd.cut(df_fe['Age'],
                           bins=[0, 12, 18, 35, 60, 100],
                           labels=['Child', 'Teen', 'Adult', 'Middle-aged', 'Senior'])

print('Survival rate by age band:')
print(df_fe.groupby('AgeBand')['Survived'].mean().round(2))

# ── 6.4 Drop columns that won't help the model ───────────────────────
df_fe.drop(columns=['Name', 'Ticket', 'PassengerId', 'Fare'], inplace=True)
print('\nFinal columns:', df_fe.columns.tolist())

---
## Section 7 — Encoding Categoricals & Scaling Numerics

ML models require **all inputs to be numbers**. Two tasks:
1. **Encode** categorical text columns → numbers
2. **Scale** numerical columns so they're on a similar range

In [ ]:
df_enc = df_fe.copy()

# ── 7.1 Binary encoding ──────────────────────────────────────────────
# Sex: male=0, female=1 (only 2 categories)
df_enc['Sex'] = (df_enc['Sex'] == 'female').astype(int)
print('Sex encoded:', df_enc['Sex'].value_counts().to_dict())

# ── 7.2 Label encoding ───────────────────────────────────────────────
# AgeBand: ordered categories → integers that preserve order
age_order = {'Child': 0, 'Teen': 1, 'Adult': 2, 'Middle-aged': 3, 'Senior': 4}
df_enc['AgeBand'] = df_enc['AgeBand'].map(age_order)

# ── 7.3 One-Hot Encoding ─────────────────────────────────────────────
# Title and Embarked: no ordinal relationship → create dummy columns
df_enc = pd.get_dummies(df_enc, columns=['Title', 'Embarked'], drop_first=True)

# Pclass is a category — one-hot encode it too
df_enc = pd.get_dummies(df_enc, columns=['Pclass'], drop_first=True)

print('\nAfter encoding, columns:')
print(df_enc.dtypes)

In [ ]:
# ── 7.4 Feature scaling ──────────────────────────────────────────────
# StandardScaler: mean=0, std=1  (best for linear models)
# MinMaxScaler:   range [0,1]    (best for neural networks)

# Convert all bool columns to int first
bool_cols = df_enc.select_dtypes(include='bool').columns
df_enc[bool_cols] = df_enc[bool_cols].astype(int)

# Ensure no remaining NaNs
df_enc.dropna(inplace=True)

cols_to_scale = ['Age', 'Fare_log', 'FamilySize']

scaler = StandardScaler()
df_enc[cols_to_scale] = scaler.fit_transform(df_enc[cols_to_scale])

print('After scaling (Age, Fare_log, FamilySize):')
print(df_enc[cols_to_scale].describe().round(2))
print(f'\nFinal dataset shape: {df_enc.shape}')
df_enc.head()

---
## Section 8 — Train/Test Split

**Never train and evaluate on the same data.** Split into a training set (model learns from it) and a test set (simulates unseen real-world data).

In [ ]:
# ── 8.1 Define X (features) and y (target) ───────────────────────────
X = df_enc.drop(columns=['Survived'])
y = df_enc['Survived']

print(f'Features (X): {X.shape}  →  {X.columns.tolist()}')
print(f'Target   (y): {y.shape}  →  {y.value_counts().to_dict()}')

# ── 8.2 Split ────────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,      # 80% train, 20% test
    random_state=42,    # reproducibility
    stratify=y          # keep same survival ratio in both splits
)

print(f'\nTraining set:  {X_train.shape[0]} rows')
print(f'Test set:      {X_test.shape[0]} rows')
print(f'Train survival rate: {y_train.mean():.2%}')
print(f'Test  survival rate: {y_test.mean():.2%}  (stratify keeps these similar)')

---
## Section 9 — Building & Evaluating a Model

Now the fun part! We'll train two models and compare them.

In [ ]:
# ── 9.1 Logistic Regression ──────────────────────────────────────────
lr = LogisticRegression(max_iter=500, random_state=42)
lr.fit(X_train, y_train)
lr_preds = lr.predict(X_test)

print('=== Logistic Regression ===')
print(f'Accuracy: {accuracy_score(y_test, lr_preds):.2%}')
print(classification_report(y_test, lr_preds, target_names=['Died', 'Survived']))

In [ ]:
# ── 9.2 Random Forest ────────────────────────────────────────────────
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
rf_preds = rf.predict(X_test)

print('=== Random Forest ===')
print(f'Accuracy: {accuracy_score(y_test, rf_preds):.2%}')
print(classification_report(y_test, rf_preds, target_names=['Died', 'Survived']))

In [ ]:
# ── 9.3 Confusion Matrix & Feature Importance ────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion matrix for Random Forest
cm = confusion_matrix(y_test, rf_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Pred: Died', 'Pred: Survived'],
            yticklabels=['True: Died', 'True: Survived'])
axes[0].set_title('Confusion Matrix — Random Forest')

# Feature importance
importances = pd.Series(rf.feature_importances_, index=X.columns)
importances.sort_values().tail(10).plot(kind='barh', ax=axes[1], color='teal')
axes[1].set_title('Top 10 Feature Importances (Random Forest)')

plt.tight_layout()
plt.show()

---
## Section 10 — End-to-End sklearn Pipeline

A **Pipeline** chains preprocessing + model into a single object. This prevents data leakage, makes deployment easy, and keeps code clean.

Best practice: **always use pipelines in production**.

In [ ]:
# Start fresh from the cleaned (pre-feature-engineering) dataset
# so we can show the full pipeline including encoding inside it

df_pipe = df_clean.copy()
df_pipe['Title'] = df_pipe['Name'].str.extract(r',\s*([^.]+)\.', expand=False).str.strip()
common_titles = ['Mr', 'Miss', 'Mrs', 'Master']
df_pipe['Title'] = df_pipe['Title'].apply(lambda t: t if t in common_titles else 'Rare')
df_pipe['FamilySize'] = df_pipe['SibSp'] + df_pipe['Parch'] + 1
df_pipe['IsAlone'] = (df_pipe['FamilySize'] == 1).astype(int)
df_pipe.drop(columns=['Name', 'Ticket', 'PassengerId', 'Fare', 'Cabin'], inplace=True, errors='ignore')
df_pipe['Pclass'] = df_pipe['Pclass'].astype(str)
df_pipe.dropna(inplace=True)

X_p = df_pipe.drop(columns=['Survived'])
y_p = df_pipe['Survived']

X_tr, X_te, y_tr, y_te = train_test_split(X_p, y_p, test_size=0.2, random_state=42, stratify=y_p)

# ── Define column groups ─────────────────────────────────────────────
numeric_features = ['Age', 'Fare_log', 'FamilySize', 'SibSp', 'Parch']
# Keep only columns that exist
numeric_features = [c for c in numeric_features if c in X_tr.columns]
categorical_features = ['Sex', 'Embarked', 'Title', 'Pclass']
categorical_features = [c for c in categorical_features if c in X_tr.columns]

# ── Preprocessing sub-pipelines ──────────────────────────────────────
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot',  OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# ── Combine with ColumnTransformer ───────────────────────────────────
preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer,  numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

# ── Full pipeline: preprocess + model ────────────────────────────────
full_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier',   RandomForestClassifier(n_estimators=100, random_state=42))
])

# ── Train & evaluate ─────────────────────────────────────────────────
full_pipeline.fit(X_tr, y_tr)
pipe_preds = full_pipeline.predict(X_te)

print('=== Full Pipeline Accuracy ===')
print(f'{accuracy_score(y_te, pipe_preds):.2%}')
print(classification_report(y_te, pipe_preds, target_names=['Died', 'Survived']))
print('\nPipeline ready to deploy! Call full_pipeline.predict(new_data) on any new passenger.')

---
## 🏆 Final Challenge — Improve the Model

Try to beat the baseline accuracy. Here are some ideas:

1. **Tune hyperparameters** — use `GridSearchCV` to find the best `n_estimators`, `max_depth` for the RandomForest
2. **Try a new model** — `GradientBoostingClassifier` or `XGBClassifier`
3. **Engineer more features** — e.g. `Fare_per_person = Fare / FamilySize`
4. **Cross-validation** — use `cross_val_score(model, X, y, cv=5)` for a more reliable accuracy estimate

In [ ]:
# YOUR CODE HERE — try to beat the baseline!
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.ensemble import GradientBoostingClassifier

# Example starter:
# param_grid = {'classifier__n_estimators': [100, 200], 'classifier__max_depth': [5, 10, None]}
# grid_search = GridSearchCV(full_pipeline, param_grid, cv=5, scoring='accuracy')
# grid_search.fit(X_tr, y_tr)
# print('Best params:', grid_search.best_params_)
# print('Best CV accuracy:', grid_search.best_score_)


---
## 📋 Summary — The Full Data-to-Model Workflow

```
RAW DATA
   │
   ├── 1. Load (pd.read_csv / read_excel / read_json)
   ├── 2. Inspect (.shape, .dtypes, .isnull(), .describe())
   ├── 3. Clean (handle nulls, fix types, drop duplicates)
   ├── 4. EDA (distributions, correlations, group-bys, plots)
   ├── 5. Feature Engineering (new columns, binning, extracting)
   ├── 6. Encode + Scale (OHE, label encoding, StandardScaler)
   ├── 7. Split (train_test_split with stratify)
   ├── 8. Train Model (fit on training set only!)
   ├── 9. Evaluate (accuracy, precision, recall, confusion matrix)
   └── 10. Pipeline (wrap everything for clean, leak-free deployment)
```

### Next steps
- **Cross-validation** (`cross_val_score`) for more reliable evaluation
- **Hyperparameter tuning** (`GridSearchCV`, `RandomizedSearchCV`)
- **Model explainability** (`SHAP`, `eli5`)
- **Handling imbalanced data** (`SMOTE`, `class_weight='balanced'`)
- **Saving your model** (`joblib.dump(model, 'model.pkl')`)

Happy modeling! 🚀